In [ ]:
!pip -q install transformers

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from IPython.display import HTML, display
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple", device=device)

In [ ]:
text = ("Sundar Pichai, the CEO of Google, met Narendra Modi in New Delhi on Monday to discuss "
        "investments in India. Microsoft and Infosys also announced a partnership in Bengaluru, "
        "while Elon Musk visited the Indian Institute of Technology in Mumbai.")
entities = ner(text)
df = pd.DataFrame(entities)[["word", "entity_group", "score", "start", "end"]]
df["score"] = df["score"].round(3)
display(df)

In [ ]:
# Highlight entities in the text
colors = {"PER": "#ffd6a5", "ORG": "#caffbf", "LOC": "#9bf6ff", "MISC": "#ffc6ff"}

def highlight(text, ents):
    out, last = "", 0
    for e in sorted(ents, key=lambda x: x["start"]):
        out += text[last:e["start"]]
        out += (f'<mark style="background:{colors.get(e["entity_group"], "#ddd")};padding:2px 5px;border-radius:4px">'
                f'{text[e["start"]:e["end"]]} <b style="font-size:10px">{e["entity_group"]}</b></mark>')
        last = e["end"]
    return out + text[last:]

display(HTML("<div style='font-size:16px;line-height:2'>" + highlight(text, entities) + "</div>"))

In [ ]:
# Entity statistics
counts = Counter(e["entity_group"] for e in entities)
print("Entity counts:", dict(counts))
plt.bar(counts.keys(), counts.values(), color="steelblue")
plt.title("Entities by type (PER = person, ORG = organisation, LOC = location, MISC)")
plt.show()

grouped = {}
for e in entities:
    grouped.setdefault(e["entity_group"], []).append(e["word"])
for k, v in grouped.items(): print(f"{k:5s}:", v)

In [ ]:
# Token-level (BIO) predictions without aggregation
raw_ner = pipeline("ner", model="dslim/bert-base-NER", device=device)
short = "Rahul works at Amazon in Hyderabad."
for t in raw_ner(short):
    print(f"{t['word']:12s} {t['entity']:8s} {t['score']:.3f}")

In [ ]:
# Try several sentences
tests = ["Apple opened a new store in Paris last week.",
         "Virat Kohli scored a century against Australia at Melbourne.",
         "The United Nations met in Geneva to talk about climate change."]
for s in tests:
    print(s)
    print("  ->", [(e["word"], e["entity_group"]) for e in ner(s)])